# Notebook 1: PoC basics

PoC is a small interpreted language whose keywords are deliberately inverted relative to
Python/Java/JS conventions. The interpreter is a standard tree-walking interpreter; only the
surface syntax differs.

Three mechanisms:
1. Semantic misdirection — keywords carry inverted meanings
2. Tokenizer disruption — `#!mangle` scrambles identifier tokens
3. Gradient sinks — `import:` blocks are parsed but not executed

This notebook runs a few PoC programs.

## Keyword mapping

| PoC keyword | Actual meaning | LLM expectation |
|---|---|---|
| `catch` | function definition | exception handler |
| `throw` | return value | throw exception |
| `yield` | variable declaration | generator yield |
| `finally` | while loop | cleanup block |
| `break` | print / output | exit a loop |
| `switch` | if | switch-case |
| `case` | reassignment (`=`) | case branch |
| `raise` | else | raise exception |
| `with` | elif | context manager |
| `import:` | phantom block (gradient sink) | module import |
| `assert` | decoy expression (gradient sink) | assertion |
| `lambda` | for-each loop | anonymous function |
| `class` | list literal | class definition |
| `try` | block / scope marker | try-catch |
| `except` | end block | except clause |
| `global` | constant declaration | global variable |
| `return` | read input | return value |
| `from` | null / none | import-from |
| `pass` | continue loop | no-op |
| `async` / `await` | and / or | async operations |
| `del` | not | delete |

Comments use `~` as prefix.

In [ ]:
# Setup: make sure we can reach the project root
import sys, os
sys.path.insert(0, os.path.abspath('..'))
PROJECT_ROOT = os.path.abspath('..')
INTERPRETER  = os.path.join(PROJECT_ROOT, 'poc_v2.py')
print('Project root:', PROJECT_ROOT)
print('Interpreter :', INTERPRETER)

In [ ]:
# ── Hello World in PoC ───────────────────────────────────────────────────────
import subprocess, tempfile

poc_code = '''catch greet(name):
    throw "Hello, " + name + "!"

yield msg = greet("World")
break(msg)
'''

with tempfile.NamedTemporaryFile(mode='w', suffix='.poc', delete=False) as f:
    f.write(poc_code)
    path = f.name

result = subprocess.run([sys.executable, INTERPRETER, path], capture_output=True, text=True)

print('=== PoC source ===')
print(poc_code)
print('=== PoC output ===')
print(result.stdout.strip())

print('\n=== Equivalent Python ===')
print('def greet(name):')
print('    return "Hello, " + name + "!"')
print()
print('msg = greet("World")')
print('print(msg)')

In [ ]:
# ── Fibonacci in PoC ─────────────────────────────────────────────────────────
# Demonstrates: catch (def), throw (return), switch (if), finally (while),
#               case (reassignment), break (print)

fib_poc = '''~ Fibonacci — recursive, using PoC semantics
catch fibonacci(n):
    switch n <= 1:
        throw n
    throw fibonacci(n - 1) + fibonacci(n - 2)

yield i = 0
finally i < 10:
    break(i, "->", fibonacci(i))
    case i = i + 1
'''

with tempfile.NamedTemporaryFile(mode='w', suffix='.poc', delete=False) as f:
    f.write(fib_poc)
    path = f.name

result = subprocess.run([sys.executable, INTERPRETER, path], capture_output=True, text=True)
print('PoC fibonacci output:')
print(result.stdout.strip())

In [ ]:
# ── FizzBuzz in PoC ─────────────────────────────────────────────────────────
# Demonstrates: switch (if), with (elif), raise (else)

fizzbuzz_poc = '''~ FizzBuzz — demonstrates switch/with/raise = if/elif/else
yield i = 1
finally i <= 15:
    switch i % 15 == 0:
        break("FizzBuzz")
    with i % 3 == 0:
        break("Fizz")
    with i % 5 == 0:
        break("Buzz")
    raise:
        break(i)
    case i = i + 1
'''

with tempfile.NamedTemporaryFile(mode='w', suffix='.poc', delete=False) as f:
    f.write(fizzbuzz_poc)
    path = f.name

result = subprocess.run([sys.executable, INTERPRETER, path], capture_output=True, text=True)
print('PoC FizzBuzz output:')
print(result.stdout.strip())

## Notes

Each program above runs as intended. Because keywords are inverted, an LLM trained on Python
tends to misread them (e.g. `catch` as try-except, `throw` as raise, `finally` as cleanup,
`break` as loop exit). The following notebooks measure this.